In [2]:
import os
import time
import uuid
import chromadb
from groq import Groq
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime

load_dotenv()

print("Day 14 - Complete RAG Pipeline")
print("All imports successful")

# Initialize core components
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Groq client: ready")
print(f"Embedder: ready")

Day 14 - Complete RAG Pipeline
All imports successful
Groq client: ready
Embedder: ready


In [ ]:
@dataclass
class RAGResponse:
    """Structured response from the RAG pipeline"""
    query: str
    answer: str
    retrieved_chunks: List[Dict]
    latency_ms: float
    total_tokens: int
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    
    def display(self):
        print(f"Query: {self.query}")
        print(f"Answer: {self.answer}")
        print(f"Sources used: {len(self.retrieved_chunks)}")
        print(f"Latency: {self.latency_ms:.0f}ms")
        print(f"Tokens: {int(self.total_tokens)}")


class EnterpriseRAGPipeline:
    """
    Complete Enterprise RAG Pipeline.
    Connects hybrid search + prompt engineering + LLM generation.
    This is the core of your resume project.
    """

    def __init__(
        self,
        org_id: str,
        groq_client: Groq,
        embedder: SentenceTransformer,
        persist_path: str = "./rag_pipeline_db",
        model: str = "llama-3.1-8b-instant",
        temperature: float=0.1,
        n_results: int =3
    ):
        self.org_id = org_id
        self.client = groq_client
        self.embedder = embedder
        self.model = model
        self.temperature = temperature
        self.n_results = n_results

        #ChromaDB
        self.chroma_client = chromadb.PersistentClient(path=persist_path)
        self.collection = self.chroma_client.get_or_create_collection(
            name=f"org_{org_id}"
        )

        # BM25
        self.bm25 = None
        self.bm25_corpus = []
        self.bm25_ids = []
        self.bm25_metadatas =[]

        # Conversation memory
        self.conversation_history = []

        # System prompt
        self.system_prompt = """You are an Enterprise RAG assistant.
Answer questions based ONLY on the provided document chunks.
Always cite which chunk you used (e.g 'According to chunk 1...').
If the answer is not in the chunks say exaxtly:
'I cannot find this information in the provided documents.'
Never make up information. Be concise and professional."""

        print(f"[{org_id}] Pipeline initialized")

    def ingest(
            self,
            texts: List[str],
            metadatas: List[Dict],
            ids: Optional[List[str]] = None
    )-> None:
        """Ingest documents into the pipeline"""
        if ids is None:
            ids = [str(uuid.uuid4())[:8] for _ in texts]

        # Add to ChromaDB
        embeddings = self.embedder.encode(texts).tolist()
        self.collection.add(
            ids=ids,
            embeddings = embeddings,
            documents =texts,
            metadatas = metadatas
        )

        # Add to BM25
        self.bm25_corpus.extend(texts)
        self.bm25_ids.extend(ids)
        self.bm25_metadatas.extend(metadatas)
        tokenized = [doc.lower().split() for doc in self.bm25_corpus]
        self.bm25 = BM25Okapi(tokenized)

        print(f"[{self.org_id}] Ingested {len(texts)} docs | Total: {len(self.bm25_corpus)}")


    def _vector_search(self, query: str, n: int) -> List[Tuple[str, float]]:
        query_embedding = self.embedder.encode(query).tolist()
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n, self.collection.count())
        )
        return [
            (results['ids'][0][i], 1 -results['distances'][0][i])
            for i in range(len(results['ids'][0]))
        ]
    
    def _bm25_search(self, query: str, n:int) -> List[Tuple[str, float]]:
        if self.bm25 is None:
            return []
        scores = self.bm25.get_scores(query.lower().split())
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:n]
        return [(self.bm25_ids[idx], score) for idx, score in ranked]
    
    def _rrf(
        self,
        result_sets: List[List[Tuple[str, float]]],
        k: int = 60
    ) -> List[Tuple[str, float]]:
        rrf_scores = {}
        for result_set in result_sets:
            for rank, (doc_id, _) in enumerate(result_set):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0/ (k+ rank+1)
        return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    def _retrieve(self, query: str)-> List[Dict]:
        """Hybrid retrieval - BM25 + vector + RRf"""
        vector_results = self._vector_search(query, self.n_results*2)
        bm25_results = self._bm25_search(query, self.n_results*2)
        merged = self._rrf([vector_results, bm25_results])[:self.n_results]

        chunks = []
        for doc_id, score in merged:
            if doc_id in self.bm25_ids:
                idx = self.bm25_ids.index(doc_id)
                chunks.append({
                    "id": doc_id,
                    "text": self.bm25_corpus[idx],
                    "score": score,
                    "metadata": self.bm25_metadatas[idx]
                })
        return chunks

    def _build_prompt(self, query:str, chunks: List[Dict]) -> str:
        """Build user message from query and chunks"""
        context_parts = []
        for i, chunk in enumerate(chunks):
            context_parts.append(
                f"Chunk {i+1} [{chunk['metadata'].get('source', 'unknown')}]:\n{chunk['text']}"
            )        
        context = "\n\n".join(context_parts)
        return f"Document chunks:\n{context}\n\nQuestion: {query}"

    def _generate(self, user_message: str) -> Tuple[str, int]:
        """Generate answer using LLM with conversation history"""
        messages = [{"role": "system", "content": self.system_prompt}]
        messages.extend(self.conversation_history)
        messages.append({"role": "user", "content": user_message})

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature = self.temperature,
            max_tokens=500
        )        
        tokens = 0
        if response.usage:
            tokens = response.usage.prompt_tokens + response.usage.completion_tokens
        return (
            response.choices[0].message.content,
            tokens
            )
    
    def query(self, user_query:str, use_memory:bool=True)-> RAGResponse:
        """
        Main query method - full RAG pipeline.
        retrieve -> build prompt -> generate -> return structured response
        """
        start = time.time()

        # Step 1 -Retrieve
        chunks = self._retrieve(user_query)

        # Step 2 - Build prompt
        user_message = self._build_prompt(user_query, chunks)

        # Step 3 - Generate
        answer, tokens = self._generate(user_message)

        # Step 4 - Update conversation history
        if use_memory:
            self.conversation_history.append(
                {"role": "user", "content": user_query}
            )
            self.conversation_history.append(
                {"role": "assistant", "content": answer}
            )
            # Keep last 6 messages (3 turns)
            self.conversation_history = self.conversation_history[-6:]
        
        latency = (time.time()- start) * 1000

        return RAGResponse(
            query=user_query,
            answer=answer,
            retrieved_chunks=chunks,
            latency_ms=latency,
            total_tokens=tokens
        )
    
    def clear_memory(self):
        """clear conversation history"""
        self.conversation_history=[]
        print(f"[{self.org_id}] Memory cleared")

    def __str__(self):
        return (f"EnterpriseRAGPipeline("
                f"org={self.org_id},"
                f"docs={len(self.bm25_corpus)},"
                f"memory={len(self.conversation_history)} messages)")
    
print("EnterpriseRAGPipeline class defined")


EnterpriseRAGPipeline class defined


In [12]:
print("=== Initializing Enterprise RAG Pipeline ===\n")

pipeline = EnterpriseRAGPipeline(
    org_id="acme_corp",
    groq_client=groq_client,
    embedder=embedder,
    persist_path="./rag_pipeline_db"
)

# Ingest a knowledge base
pipeline.ingest(
    texts=[
        "Hybrid search combines BM25 keyword search with vector semantic search. Results are merged using Reciprocal Rank Fusion (RRF) which uses rank position not raw scores.",
        "RAGAs evaluates RAG pipelines using four metrics: faithfulness measures if answers are grounded in context, answer relevancy measures if the answer addresses the question.",
        "Multi-tenancy is implemented through ChromaDB collection isolation. Each organization gets a separate collection — org_acme_corp, org_techco etc. Users cannot access other organizations data.",
        "Re-ranking uses a cross-encoder model to re-score retrieved chunks. Unlike bi-encoders, cross-encoders process query and document together for higher precision.",
        "LoRA fine-tuning reduces trainable parameters by 90 percent using low-rank matrix decomposition. It adds small adapter matrices to existing weights instead of modifying them.",
        "FastAPI provides automatic OpenAPI documentation, async request handling, and Pydantic data validation. It is the recommended framework for building ML backends.",
        "Docker containerization ensures consistent environments. docker-compose manages multiple services including FastAPI, ChromaDB, Redis and Celery workers.",
        "The pipeline is deployed on AWS EC2 with Nginx as reverse proxy. GitHub Actions handles CI/CD — auto deploys on push to main branch.",
        "Langfuse provides LLM observability — traces every call showing retrieved chunks, tokens used, latency and model parameters.",
        "Celery with Redis handles async document ingestion. Large PDFs process in background — user gets a job_id to poll status instead of waiting."
    ],
    metadatas=[
        {"source": "architecture.pdf", "topic": "retieval"},
        {"source": "evaluation.pdf", "topic": "evaluation"},
        {"source": "architecture.pdf", "topic": "multitenancy"},
        {"source": "retrieval.pdf", "topic": "reranking"},
        {"source": "training.pdf", "topic": "finetuning"},
        {"source": "backend.pdf", "topic": "api"},
        {"source": "deployment.pdf", "topic": "docker"},
        {"source": "deployment.pdf", "topic": "cloud"},
        {"source": "monitoring.pdf", "topic": "observability"},
        {"source": "backend.pdf", "topic": "async"}
    ]
)

print(f"\n{pipeline}")

=== Initializing Enterprise RAG Pipeline ===

[acme_corp] Pipeline initialized
[acme_corp] Ingested 10 docs | Total: 10

EnterpriseRAGPipeline(org=acme_corp,docs=10,memory=0 messages)


In [9]:
print("=== Testing RAG Pipeline ===\n")

# Query 1 - retrieval question
print("=" * 55)
r1 = pipeline.query("How does hybrid search work?")
r1.display()

# Query 2 -evaluation
print("\n"+"=" * 55)
r2 = pipeline.query("What metrics does RAGAs use?")
r2.display()

# Query 3 - follow up (tests memory)
print("\n"+"=" * 55)
r3 = pipeline.query("Which of those metrics is most import?")
r3.display()

# Query 4 - unkown question (tests I don't Know)
print("\n"+"=" * 55)
r4 = pipeline.query("What is the pricing of the Enterprise plan?")
r4.display()

# Query 5 - exact keyword (tests BM25)
print("\n"+"=" * 55)
r5 = pipeline.query("How does LoRA reduce trainable parameters?")
r5.display()

=== Testing RAG Pipeline ===

Query: How does hybrid search work?
Answer: According to chunk 1, hybrid search combines BM25 keyword search with vector semantic search. The results are then merged using Reciprocal Rank Fusion (RRF).
Sources used: 3
Latency: 421ms
Tokens: 0

Query: What metrics does RAGAs use?
Answer: According to chunk 1, RAGAs evaluates RAG pipelines using two metrics: 
1. Faithfulness 
2. Answer relevancy.
Sources used: 2
Latency: 217ms
Tokens: 0

Query: Which of those metrics is most import?
Answer: I cannot find this information in the provided documents.
Sources used: 2
Latency: 175ms
Tokens: 0

Query: What is the pricing of the Enterprise plan?
Answer: I cannot find this information in the provided documents.
Sources used: 3
Latency: 195ms
Tokens: 0

Query: How does LoRA reduce trainable parameters?
Answer: According to chunk 1, LoRA reduces trainable parameters by 90 percent using low-rank matrix decomposition.
Sources used: 2
Latency: 249ms
Tokens: 0


In [13]:
print("=== Pipeline Performance Summary ===\n")

queries = [
    "How does hybrod search work?",
    "What metrics does RAGAs use?",
    "How is multi-tenancy implemented?",
    "What does Celery handle in the pipeline?",
    "How is the pipeline deployed?"
]

latencies = []
token_counts = []

for query in queries:
    response = pipeline.query(query, use_memory=False)
    latencies.append(response.latency_ms)
    token_counts.append(response.total_tokens)
    print(f"Q: {query[:50]}")
    print(f"  Latency : {response.latency_ms:.0f}ms | Tokens: {int(response.total_tokens)}")
    print(f"  A: {response.answer[:80]}...\n")

print("="*50)
print(f"Avg latency:    {sum(latencies)/len(latencies):.0f}ms")
print(f"Max latency:    {max(latencies):.0f}ms")
print(f"Min latency:    {min(latencies):.0f}ms")
print(f"Avg tokens:    {int(sum(token_counts)/len(token_counts))}")
print(f"Total tokens:    {int(sum(token_counts))}")
print(f"\nPipeline: {pipeline}")

=== Pipeline Performance Summary ===

Q: How does hybrod search work?
  Latency : 287ms | Tokens: 221
  A: According to chunk 1, hybrid search combines BM25 keyword search with vector sem...

Q: What metrics does RAGAs use?
  Latency : 268ms | Tokens: 183
  A: According to chunk 1, RAGAs evaluates RAG pipelines using four metrics: faithful...

Q: How is multi-tenancy implemented?
  Latency : 318ms | Tokens: 224
  A: According to chunk 1, multi-tenancy is implemented through ChromaDB collection i...

Q: What does Celery handle in the pipeline?
  Latency : 173ms | Tokens: 210
  A: According to chunk 1, Celery with Redis handles async document ingestion....

Q: How is the pipeline deployed?
  Latency : 165ms | Tokens: 179
  A: According to chunk 1, the pipeline is deployed on AWS EC2 with Nginx as reverse ...

Avg latency:    242ms
Max latency:    318ms
Min latency:    165ms
Avg tokens:    203
Total tokens:    1017

Pipeline: EnterpriseRAGPipeline(org=acme_corp,docs=10,memory=0 messages)
